# Text + subreddits

Run the text baseline first. This notebook joins text TF-IDF and subreddit
TF-IDF, then checks whether the combined model improves on text alone.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

import pandas as pd
from scipy.stats import pearsonr
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction import DictVectorizer
from sklearn.feature_extraction.text import TfidfTransformer, TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import make_pipeline

DATA = Path("../data/preprocessed")  # From the project's src folder
TEXT_RESULTS = Path("../data/text_baseline/predictions.csv")
OUTPUT = Path("../data/combined_baseline")
TRAITS = ["agreeableness", "openness", "conscientiousness",
          "extraversion", "neuroticism"]
ALPHAS = [0.01, 0.1, 1, 10, 100]

## 1. Get text and counts for each user

Use the same users and folds as in the two baselines.

In [ ]:
users = pd.read_csv(DATA / "main/users.csv").set_index("author")
outer = pd.read_csv(DATA / "outer_folds.csv").set_index("author")
inner = pd.read_csv(DATA / "inner_folds.csv")
assert users.index.is_unique and outer.index.is_unique
assert set(users.index) == set(outer.index)

comments = pd.read_csv(DATA / "main/comments.csv", usecols=["author", "body"],
                       keep_default_na=False)
documents = comments.groupby("author")["body"].agg(" ".join)
del comments

data = users[TRAITS].join(outer).join(documents.rename("text"))
counts = pd.read_csv(DATA / "main/subreddit_counts.csv")
subreddit_counts = {}
for author, rows in counts.groupby("author"):
    subreddit_counts[author] = dict(zip(rows["subreddit"], rows["comment_count"]))

data["counts"] = data.index.map(subreddit_counts)
assert data[["text", "counts"]].notna().all().all()
print("Users:", len(data))

## 2. Train the combined model

Fit both TF-IDF steps on the training users. Join the two feature matrices
and use the inner folds to choose alpha for each trait.

In [ ]:
predictions = []

for fold in sorted(data["outer_fold"].unique()):
    train = data[data["outer_fold"] != fold]
    test = data[data["outer_fold"] == fold]

    inner_fold = inner[inner["outer_fold"] == fold].set_index("author")["inner_fold"]
    assert inner_fold.index.is_unique and set(inner_fold.index) == set(train.index)
    inner_fold = inner_fold.reindex(train.index).to_numpy()
    cv = []
    for val_fold in sorted(set(inner_fold)):
        train_idx = (inner_fold != val_fold).nonzero()[0]
        val_idx = (inner_fold == val_fold).nonzero()[0]
        cv.append((train_idx, val_idx))

    features = ColumnTransformer([
        ("text", TfidfVectorizer(min_df=2, max_features=50_000), "text"),
        ("subreddits", make_pipeline(DictVectorizer(), TfidfTransformer()), "counts"),
    ], sparse_threshold=1.0)

    # Reuse training features across the Ridge searches.
    with TemporaryDirectory() as cache:
        model = make_pipeline(
            features,
            Ridge(solver="lsqr"),
            memory=cache,
        )

        for trait in TRAITS:
            search = GridSearchCV(model, {"ridge__alpha": ALPHAS}, cv=cv,
                                  scoring="neg_root_mean_squared_error")
            search.fit(train[["text", "counts"]], train[trait])
            predictions.append(pd.DataFrame({
                "author": test.index,
                "outer_fold": fold,
                "trait": trait,
                "true": test[trait].to_numpy(),
                "combined_prediction": search.predict(test[["text", "counts"]]),
                "alpha": search.best_params_["ridge__alpha"],
            }))
    print("Finished fold", fold)

## 3. Compare with text alone

Match predictions by user, trait, and test fold. Negative RMSE change means
the combined model made smaller errors.

In [ ]:
results = pd.concat(predictions, ignore_index=True)
assert len(results) == len(users) * len(TRAITS)
assert not results.duplicated(["author", "trait"]).any()

text_results = pd.read_csv(TEXT_RESULTS)
keys = ["author", "trait", "outer_fold"]
paired = results.merge(
    text_results[keys + ["true", "text_prediction"]],
    on=keys, validate="one_to_one", suffixes=("", "_text"),
)
assert len(paired) == len(results) == len(text_results)
assert paired["true"].eq(paired["true_text"]).all()

metrics = []
for trait, rows in paired.groupby("trait"):
    actual = rows["true"]
    for model in ["text", "combined"]:
        predicted = rows[f"{model}_prediction"]
        correlation = float("nan")
        if predicted.nunique() > 1 and actual.nunique() > 1:
            correlation = pearsonr(actual, predicted)[0]
        metrics.append({
            "trait": trait,
            "model": model,
            "rmse": mean_squared_error(actual, predicted) ** 0.5,
            "pearson_r": correlation,
            "r2": r2_score(actual, predicted),
        })

metrics = pd.DataFrame(metrics)
OUTPUT.mkdir(parents=True, exist_ok=True)
results.to_csv(OUTPUT / "predictions.csv", index=False)
metrics.to_csv(OUTPUT / "metrics.csv", index=False)
comparison = metrics.pivot(index="trait", columns="model", values="rmse")
comparison["rmse_change"] = comparison["combined"] - comparison["text"]
comparison.to_csv(OUTPUT / "comparison.csv")
display(comparison.round(3))
display(metrics.round(3))